# Phase 1: Targeted Data Cleaning & Preprocessing
This notebook executes the data cleaning pipeline mapped for E-Commerce edge cases.

## Initialization and Loading Data
Load the raw dataset.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("Loading dataset...")
df = pd.read_excel("DailySales_original.xlsx")
print(f"Original shape: {df.shape}")

Loading dataset...
Original shape: (13324, 37)


## 1. Drop the Grand Total Row
Identify and drop the final aggregate row to prevent data duplication.

In [2]:
# Dropping the Grand Total row (indicated by missing Date)
df = df.dropna(subset=["Date"])
print(f"Shape after dropping Grand Total: {df.shape}")

Shape after dropping Grand Total: (13323, 37)


## Basic Cleaning
Dropping columns heavily skewed with nulls that provide no analytical value.

In [ ]:
useless_cols = ["Internal Name", "Product Group", "FNSKU", "Is Parent"]
df = df.drop(columns=[col for col in useless_cols if col in df.columns], errors="ignore")
print("Dropped extraneous columns.")

## 2. Smart Imputation for Missing Values
Handling heavy missing values intelligently rather than using crude `.fillna(0)`.

### A. Parent ASIN & Brand/Title Lookup
Fill heavy nulls by mapping them to their base `ASIN` and mapping `Title`/`Brand` from dates where data is present.

In [4]:
# Fill Parent ASIN with ASIN if missing
if "Parent ASIN" in df.columns and "ASIN" in df.columns:
    df["Parent ASIN"] = df["Parent ASIN"].fillna(df["ASIN"])

# Map missing strings via SKU references
if "SKU" in df.columns:
    title_map = df.dropna(subset=["Title"]).groupby("SKU")["Title"].first()
    brand_map = df.dropna(subset=["Brand"]).groupby("SKU")["Brand"].first()
    
    if "Title" in df.columns:
        df["Title"] = df["Title"].fillna(df["SKU"].map(title_map)).fillna("Unknown Title")
    if "Brand" in df.columns:
        df["Brand"] = df["Brand"].fillna(df["SKU"].map(brand_map)).fillna("Unknown Brand")
print("Parent ASIN and Categorical map imputed.")

Parent ASIN and Categorical map imputed.


### B. Per Unit Revenue Imputation
Calculate historical average price for specific SKUs, or set to 0 if 0 orders are placed.

In [5]:
if "Per Unit Revenue" in df.columns and "Orders" in df.columns:
    # Calculate historical average price for specific SKU
    avg_price_map = df[df["Per Unit Revenue"] > 0].groupby("SKU")["Per Unit Revenue"].mean()
    
    # Fill with historical average first
    df["Per Unit Revenue"] = df["Per Unit Revenue"].fillna(df["SKU"].map(avg_price_map))
    
    # Failsafe to 0 for missing limits and 0 orders
    df["Per Unit Revenue"] = np.where(
        df["Per Unit Revenue"].isna() & (df["Orders"] == 0),
        0,
        df["Per Unit Revenue"]
    )
    df["Per Unit Revenue"] = df["Per Unit Revenue"].fillna(0)
print("Per Unit Revenue securely imputed.")

Per Unit Revenue securely imputed.


## 3. Formatting & Export
Formatting dates and deriving structural Gross Margins before saving.

In [6]:
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(0)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

if "Net Profit" in df.columns and "Revenue" in df.columns:
    df["Gross Margin"] = df["Revenue"] - df["COGS"].abs()

output_file = "DailySales_cleaned_professional.xlsx"
df.to_excel(output_file, index=False)
print(f"Fully cleaned dataset saved to {output_file}")

Fully cleaned dataset saved to DailySales_cleaned_professional.xlsx
